Adverserial resistant Fake payment Gateway detection using transactions patterns
This project detects fake payment gateways using URL metadata, transaction patterns, and adversarial-resistant feature engineering.

**Data sets Collection (Without Adverserial)**
#
Adverserial is for fooling a model as attacker can be able to bypass

In [25]:
import pandas as pd
df = pd.read_csv('../raw/adverserial_legit.csv')
print(df)

                         url  label  TransactionAmount TransactionType  \
0          http://paypal.com      1               4.99           Debit   
1      http://google pay.com      0              50.00          Credit   
2       http://apple pay.com      1               1.99           Debit   
3          http://stripe.com      0             120.50           Debit   
4  https://bankofamerica.com      1               9.99           Debit   
5          http://Alipay.com      0               3.00          Credit   
6          http://skrill.com      1               0.49           Debit   
7           http://wepay.com      0              75.00          Credit   

   CustomerAge  AccountBalance    Location  
0           35          1200.0      Sylhet  
1           45          5000.0       Dhaka  
2           22           500.0  Chittagong  
3           55         15000.0    Rajshahi  
4           60           300.0      Khulna  
5           30          7000.0       Dhaka  
6           19    

**Adverserial Generations**
#
interms of there homoglyph_domain,add_https,mimic_timing,age_spoof

In [ ]:
import pandas as pd
import random
from urllib.parse import urlparse
import os


def homoglyph_domain(url):
    p = urlparse(url)
    host = p.netloc
    host2 = host.replace('l','I').replace('o','0').replace('a','@')
    return url.replace(host, host2)

def add_https(url):
    if url.startswith('http://'):
        return url.replace('http://', 'https://')
    return url

def mimic_timing(row):
    return random.uniform(5.0, 15.0)

def age_spoof(row):
    return random.choice([15, 30, 60, 90])

def generate_variants(df, n_variants=5):
    adv_rows = []

    phish_targets = df[df['label'] == 1]

    required_cols = list(df.columns) + [
        'notes', 'num_redirects', 'https_flag',
        'domain_age_days', 'time_to_confirm', 'page_dwell'
    ]

    for _, row in phish_targets.iterrows():
        for i in range(n_variants):
            r = row.copy()

            
            for col in ['notes', 'num_redirects', 'https_flag',
                        'domain_age_days', 'time_to_confirm', 'page_dwell']:
                r[col] = None

            attack = random.choice([
                'homoglyph', 'https', 'timing',
                'age', 'redirects', 'amount_mimic'
            ])

            if attack == 'homoglyph':
                r['url'] = homoglyph_domain(r['url'])
                r['notes'] = 'homoglyph'

            elif attack == 'https':
                r['url'] = add_https(r['url'])
                r['https_flag'] = 1
                r['notes'] = 'https_added'

            elif attack == 'timing':
                r['time_to_confirm'] = mimic_timing(row)
                r['page_dwell'] = max(0.5, random.uniform(0.7, 1.5))
                r['notes'] = 'timing_mimic'

            elif attack == 'age':
                r['domain_age_days'] = age_spoof(row)
                r['notes'] = 'age_spoof'

            elif attack == 'redirects':
                r['num_redirects'] = random.randint(3, 7)
                r['notes'] = 'more_redirects'

            elif attack == 'amount_mimic':
                r['TransactionAmount'] = random.choice([0.49, 1.99, 5.0, 9.99])
                r['notes'] = 'amount_mimic'

            adv_rows.append(r)

    adv_df = pd.DataFrame(adv_rows)
    adv_df = adv_df.reindex(columns=required_cols)

    
    
    adv_df = adv_df.dropna(axis=1, how='any')

    
    adv_df = adv_df.dropna(axis=0, how='any')
    

    return adv_df


if __name__ == "__main__":
    os.makedirs('../processed', exist_ok=True)
    os.makedirs('../raw', exist_ok=True)

    input_path = "../raw/adverserial_legit.csv"
    output_path = "../processed/sessions_with_adv.csv"

    try:
        manual_df = pd.read_csv(input_path)
    except FileNotFoundError:
        print(f"Error: {input_path} not found. Please create it with the updated structure first.")
        exit()

    print(f"Generating adversarial variants from {len(manual_df[manual_df['label'] == 1])} targets...")

    adv_test_df = generate_variants(manual_df)

    adv_test_df.to_csv(output_path, index=False)

    print(f"Saved {len(adv_test_df)} adversarial test samples to {output_path}")


Generating adversarial variants from 4 targets...
Saved 20 adversarial test samples to ../processed/sessions_with_adv.csv


**Dataset(Adverserial Generated)** 
#
Adverserial dataset created for model so that model understand adverserial datasets 


In [36]:
import pandas as pd
df = pd.read_csv('../processed/sessions_with_adv.csv')
print(df.head())

                  url  label  TransactionAmount TransactionType  CustomerAge  \
0   http://paypal.com      1               4.99           Debit           35   
1   http://p@yp@I.c0m      1               4.99           Debit           35   
2  https://paypal.com      1               4.99           Debit           35   
3   http://p@yp@I.c0m      1               4.99           Debit           35   
4   http://paypal.com      1               4.99           Debit           35   

   AccountBalance Location           notes  
0          1200.0   Sylhet       age_spoof  
1          1200.0   Sylhet       homoglyph  
2          1200.0   Sylhet     https_added  
3          1200.0   Sylhet       homoglyph  
4          1200.0   Sylhet  more_redirects  


**Dataset(Transaction Patterns normal bank transaction data)**

In [9]:
import pandas as pd
df = pd.read_csv('../raw/bank_transactions_data_kaggle.csv')
print(df.head())

  TransactionID AccountID  TransactionAmount      TransactionDate  \
0      TX000001   AC00128              14.09  2023-04-11 16:29:14   
1      TX000002   AC00455             376.24  2023-06-27 16:44:19   
2      TX000003   AC00019             126.29  2023-07-10 18:16:08   
3      TX000004   AC00070             184.50  2023-05-05 16:32:11   
4      TX000005   AC00411              13.45  2023-10-16 17:51:24   

  TransactionType   Location DeviceID      IP Address MerchantID Channel  \
0           Debit  San Diego  D000380  162.198.218.92       M015     ATM   
1           Debit    Houston  D000051     13.149.61.4       M052     ATM   
2           Debit       Mesa  D000235  215.97.143.157       M009  Online   
3           Debit    Raleigh  D000187  200.13.225.150       M002  Online   
4          Credit    Atlanta  D000308    65.164.3.100       M091  Online   

   CustomerAge CustomerOccupation  TransactionDuration  LoginAttempts  \
0           70             Doctor                   81 

In [8]:
import pandas as pd
import random

input_path = "../raw/bank_transactions_data_kaggle.csv"
output_path = "../processed/kaggle_legit.csv"

bd_cities = ["Dhaka", "Chittagong", "Sylhet", "Rajshahi", "Khulna"]

print("Loading Kaggle dataset...")
df = pd.read_csv(input_path)

df_small = df.head(100)  

df_small["Location"] = [random.choice(bd_cities) for _ in range(len(df_small))]
df_small["label"] = 0  

df_small.to_csv(output_path, index=False)

print("Saved Kaggle legitimate dataset to data/raw/kaggle_legit.csv")


Loading Kaggle dataset...
Saved Kaggle legitimate dataset to data/raw/kaggle_legit.csv


C:\Users\Sadrib\AppData\Local\Temp\ipykernel_7680\2390803213.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_small["Location"] = [random.choice(bd_cities) for _ in range(len(df_small))]
C:\Users\Sadrib\AppData\Local\Temp\ipykernel_7680\2390803213.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_small["label"] = 0


Dataset(Legit Transactions Pattern with locations changed)

In [9]:
import pandas as pd
df = pd.read_csv('../processed/kaggle_legit.csv')
print(df.head())

  TransactionID AccountID  TransactionAmount      TransactionDate  \
0      TX000001   AC00128              14.09  2023-04-11 16:29:14   
1      TX000002   AC00455             376.24  2023-06-27 16:44:19   
2      TX000003   AC00019             126.29  2023-07-10 18:16:08   
3      TX000004   AC00070             184.50  2023-05-05 16:32:11   
4      TX000005   AC00411              13.45  2023-10-16 17:51:24   

  TransactionType  Location DeviceID      IP Address MerchantID Channel  \
0           Debit    Sylhet  D000380  162.198.218.92       M015     ATM   
1           Debit     Dhaka  D000051     13.149.61.4       M052     ATM   
2           Debit  Rajshahi  D000235  215.97.143.157       M009  Online   
3           Debit     Dhaka  D000187  200.13.225.150       M002  Online   
4          Credit     Dhaka  D000308    65.164.3.100       M091  Online   

   CustomerAge CustomerOccupation  TransactionDuration  LoginAttempts  \
0           70             Doctor                   81       

**Datasets(Transactions patterns phishing using phishtank )**

In [3]:
import pandas as pd
df = pd.read_csv('../raw/phishtank_urls.csv')
print(df.head())

   Unnamed: 0 FirstName     LastName         Dob Gender    UserId  Status  \
0           0       smt  shyani devi  28-05-1985      F  13260350       1   
1           1   shakshi        sagar  09-06-1998      F  13454400       1   
2           2     anshu          d/o  23-11-2001      F  13321313       1   
3           3    kanika     kathuria  20-06-2001      F  12444655       1   
4           4      riya         masi  18-07-2005      U  12547615       1   

                 Email      Mobile  ProgramId  ... PresentLoginTime  \
0    smtp123@gmail.com  8812569734       6019  ...         19:34:34   
1   shakiiit@gmail.com  8615389304       6019  ...         18:58:54   
2   ansshhhh@gmail.com  8418054019       6019  ...         18:44:38   
3  kankik123@gmail.com  8055221226       6019  ...         18:41:04   
4     riyaaa@gmail.com  8556603655       6019  ...         18:08:58   

         RegisteredOn  OS Type Reg Referral Code Reg Referral Prefix  \
0  10/19/2020   18:45  Android        

In [4]:
import pandas as pd
import os


input_path = "../raw/phishtank_urls.csv"
output_path = "../processed/phishing.csv"


os.makedirs(os.path.dirname(output_path), exist_ok=True)

print(f"Loading PhishTank data from: {input_path}")

try:
    
    df_phish = pd.read_csv(input_path)
    df_phish = df_phish.head(100)  

    
    df_phish["label"] = 1

    
    df_phish.to_csv(output_path, index=False)

    print(f"Successfully processed {len(df_phish)} phishing URLs.")
    print(f"Saved processed phishing dataset to: {output_path}")

except FileNotFoundError:
    print(f"Error: The input file {input_path} was not found.")
    print("Please make sure you have run the data collection script (`collect_phishtank.py`) first.")
except Exception as e:
    print(f"An error occurred during processing: {e}")

Loading PhishTank data from: ../raw/phishtank_urls.csv
Successfully processed 100 phishing URLs.
Saved processed phishing dataset to: ../processed/phishing.csv


**Data preprocessing** 
#
After we label all datasets This step it's about cleaning transforming and preparing data so it's high quality and ready for modeling
#

In [12]:
import pandas as pd
import tldextract
import os

def extract_domain_features(url):
    ext = tldextract.extract(url)
    return ext.domain, ext.suffix

print("\n--- Running Preprocessing Pipeline ---\n")

try:
    phishing = pd.read_csv("../processed/phishing.csv")
    kaggle_legit = pd.read_csv("../processed/kaggle_legit.csv")
    adversarial = pd.read_csv("../processed/sessions_with_adv.csv")
except FileNotFoundError as e:
    print(f"File not found: {e}")
    exit()

phishing["label"] = 1
kaggle_legit["label"] = 0

if "label" not in adversarial.columns:
    adversarial["label"] = 0

required_cols = [
    "url",
    "TransactionAmount",
    "TransactionType",
    "CustomerAge",
    "AccountBalance",
    "Location",
    "Email",
    "label"
]

def ensure_columns(df):
    for col in required_cols:
        if col not in df.columns:
            df[col] = None
    return df[required_cols]

phishing = ensure_columns(phishing)
kaggle_legit = ensure_columns(kaggle_legit)
adversarial = ensure_columns(adversarial)

df = pd.concat([phishing, kaggle_legit, adversarial], ignore_index=True).drop_duplicates(subset=["url"])

print(f"Dataset size before cleaning: {len(df)}")
df.dropna(subset=["url"], inplace=True) 


if "Email" in df.columns:
    df.drop("Email", axis=1, inplace=True)
    print("Dropped the 'Email' column due to excessive missing values.")


print(f"Dataset size after cleaning: {len(df)}")

output_path = "../processed/preprocessed_dataset.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}\n")


--- Running Preprocessing Pipeline ---

Dataset size before cleaning: 9
Dropped the 'Email' column due to excessive missing values.
Dataset size after cleaning: 8
Saved cleaned dataset to: ../processed/preprocessed_dataset.csv



C:\Users\Sadrib\AppData\Local\Temp\ipykernel_7680\4233239999.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([phishing, kaggle_legit, adversarial], ignore_index=True).drop_duplicates(subset=["url"])


**Data processing Output**
# 
Merged and processed labelled data of transactions patterns, adverserial and fake detections

In [13]:
import pandas as pd
df = pd.read_csv('../processed/preprocessed_dataset.csv')
print(df)


                         url  TransactionAmount TransactionType  CustomerAge  \
0          http://paypal.com               4.99           Debit           35   
1          http://p@yp@I.c0m               4.99           Debit           35   
2       http://apple pay.com               1.99           Debit           22   
3       http://@ppIe p@y.c0m               1.99           Debit           22   
4  https://bankofamerica.com               1.99           Debit           60   
5  https://b@nk0f@meric@.c0m               9.99           Debit           60   
6          http://skrill.com               0.49           Debit           19   
7         https://skrill.com               0.49           Debit           19   

   AccountBalance    Location  label  
0          1200.0      Sylhet      1  
1          1200.0      Sylhet      1  
2           500.0  Chittagong      1  
3           500.0  Chittagong      1  
4           300.0      Khulna      1  
5           300.0      Khulna      1  
6     

**Feature Engineering**
# 
This section creates advanced, robust features (like character entropy, domain length, etc.) from the combined dataset, preparing it for adverserial resistant and detecting fake payment detection approaches to the selected model.


In [1]:
import pandas as pd
from urllib.parse import urlparse
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
import os

def calculate_entropy(s):
    if not isinstance(s, str) or not s:
        return 0.0
    freq = {}
    ent = 0.0
    for c in s:
        freq[c] = freq.get(c, 0) + 1
    for v in freq.values():
        p = v / len(s)
        ent += -p * np.log2(p) if p > 0 else 0
    return ent

def create_engineered_features(df):
    df['url_length'] = df['url'].apply(len)
    df['path_length'] = df['url'].apply(lambda u: len(urlparse(u).path))
    df['ssl_flag'] = df['url'].apply(lambda u: 1 if u.startswith("https") else 0)
    df['domain_entropy'] = df['domain'].apply(calculate_entropy)

    ip_pattern = r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
    df['uses_ip'] = df['url'].apply(lambda u: 1 if re.search(ip_pattern, u) else 0)

    df['num_dots'] = df['url'].apply(lambda u: u.count('.'))
    df['uses_at_symbol'] = df['url'].apply(lambda u: 1 if '@' in u else 0)

    if 'DomainAge' in df.columns:
        df['DomainAge'] = pd.to_numeric(df['DomainAge'], errors='coerce').fillna(df['DomainAge'].mean())
    else:
        df['DomainAge'] = 0

    if 'PaymentDelay' in df.columns:
        df['PaymentDelay'] = pd.to_numeric(df['PaymentDelay'], errors='coerce').fillna(df['PaymentDelay'].mean())
    else:
        df['PaymentDelay'] = 0

    num_cols = ['TransactionAmount', 'CustomerAge', 'AccountBalance']
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].mean())

    if 'Location' in df.columns:
        le = LabelEncoder()
        df['Location_Encoded'] = le.fit_transform(df['Location'].astype(str))
    else:
        df['Location_Encoded'] = 0

    if 'TransactionType' in df.columns:
        df = pd.get_dummies(df, columns=['TransactionType'], prefix='Txn')

    if 'IP Address' in df.columns and 'Location' in df.columns:
        df['IP_mismatch'] = df.apply(
            lambda row: 1 if str(row['Location']).lower() not in str(row['IP Address']).lower() else 0, axis=1
        )
    else:
        df['IP_mismatch'] = 0

    drop_cols = [
        'url','domain','tld','Location','TransactionID','TransactionDate',
        'TransactionDuration','DeviceID','IP Address','MerchantID',
        'Channel','CustomerOccupation','LoginAttempts','PreviousTransactionDate'
    ]

    df_final = df.drop(columns=drop_cols, errors='ignore')

    for c in df_final.columns:
        df_final[c] = pd.to_numeric(df_final[c], errors='coerce').fillna(0)

    return df_final

if __name__ == "__main__":
    input_path = "../processed/preprocessed_dataset.csv"
    output_path = "../processed/Feature.csv"

    df = pd.read_csv(input_path)

    df.to_csv(output_path, index=False)
    print(f"Saved final engineered dataset to {output_path}")


Saved final engineered dataset to ../processed/Feature.csv


**Feature engineering output** 
#
Final output that is going to model the ultimate added featured

In [1]:
import pandas as pd
df = pd.read_csv('../processed/Feature.csv')
print(df)


                         url  TransactionAmount TransactionType  CustomerAge  \
0          http://paypal.com               4.99           Debit           35   
1          http://p@yp@I.c0m               4.99           Debit           35   
2       http://apple pay.com               1.99           Debit           22   
3       http://@ppIe p@y.c0m               1.99           Debit           22   
4  https://bankofamerica.com               1.99           Debit           60   
5  https://b@nk0f@meric@.c0m               9.99           Debit           60   
6          http://skrill.com               0.49           Debit           19   
7         https://skrill.com               0.49           Debit           19   

   AccountBalance    Location  label  
0          1200.0      Sylhet      1  
1          1200.0      Sylhet      1  
2           500.0  Chittagong      1  
3           500.0  Chittagong      1  
4           300.0      Khulna      1  
5           300.0      Khulna      1  
6     